# deepchunk_M20 — nới M từ 10 lên 20, KHÔNG chấm lại phần đã chấm

**Ước ~40 phút.** GPU T4, Internet On, Save & Run All.

Lượt 15/08 với M=10 cho **0.9017** (base 0.8883, +1,34). Chẩn đoán phần còn hụt:

| Nhóm | Gold | % |
|---|---|---|
| Đã vào top-5 | 281 | 89,5% |
| **Hạng 11+, CHƯA chấm sâu (ngoài M=10)** | **16** | **5,1%** |
| Hạng 6-10, đã chấm sâu — 20 đoạn vẫn chưa đủ | 10 | 3,2% |
| Ngoài top-100 BM25 | 7 | 2,2% |

Nhóm 16 câu là lớn nhất và nguyên nhân đã rõ: chúng chưa từng được chấm sâu, lại
phải đấu với nhóm hạng 1-10 đã được cộng điểm từ 20 đoạn. Nới M lên 20 vừa cho
chúng cơ hội vừa xoá bất đối xứng đó.

**Vì sao chỉ ~40 phút thay vì ~1h50m:** tầng 1 (47.240 đoạn) và `ce_deep` của hạng
1-10 đã có sẵn từ lượt trước, upload lên dataset là dùng lại được. Lượt này chỉ chấm
**hạng 11-20** (~57.600 đoạn). Đây là phần thưởng của quy tắc 2 — luôn tải scores về.

**Rủi ro cần đo:** M gấp đôi nghĩa là gấp đôi số văn bản được nâng điểm, kể cả văn
bản SAI. Tỷ lệ "cứu 5 hỏng 1" có thể xấu đi. Đó là lý do đo dev300 trước, không
nhảy thẳng vào đề thi.

In [ ]:
!pip install -q sentence-transformers

import os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
INPUT_DIR = "/kaggle/input/project-ir"
sys.path.append(INPUT_DIR)
!ls /kaggle/input

In [ ]:
import json, time
from pathlib import Path

from metrics import evaluate
from rerank import load_reranker
from rerank_from_d import blend_bm25_first
import deep_chunk as DC

## Config

In [ ]:
RERANKER_MODEL = "AITeamVN/Vietnamese_Reranker"
DEVICE = "cuda"

DEV_GOLD = f"{INPUT_DIR}/dev_300_locked.json"
DEV_CAND = f"{INPUT_DIR}/bm25_top100_dev300.json"
CTX_DIR  = f"{INPUT_DIR}/selected-contexts"

# Scores của lượt trước — PHẢI upload lên dataset, không có là hỏng cả ý tưởng
SCORES_M10 = f"{INPUT_DIR}/scores_dev300_deep_M10_K20.json"

M_OLD, M_NEW, K_CHUNK = 10, 20, 20   # chấm hạng M_OLD..M_NEW
TOPK = 5

BASE_LINE = 0.8883    # AITeamVN gốc + blend n=2
M10_BEST  = 0.9017    # max + n=2, lượt 15/08
OUTPUT_DIR = "/kaggle/working/outputs"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## Bước 1 — Nạp lại scores lượt trước

Nếu cell này báo thiếu file thì **dừng**: chạy tiếp sẽ phải chấm lại từ đầu,
mất 30 phút vô ích. Upload `scores_dev300_deep_M10_K20.json` lên dataset rồi quay lại.

In [ ]:
dev  = json.load(open(DEV_GOLD, encoding="utf-8"))
cand = json.load(open(DEV_CAND, encoding="utf-8"))
dev_q    = {q: v["question"] for q, v in dev.items()}
dev_gold = {q: v["answer"]   for q, v in dev.items()}

assert os.path.exists(SCORES_M10), f"THIẾU {SCORES_M10} — upload lên dataset trước đã"
scores = json.load(open(SCORES_M10, encoding="utf-8"))
assert set(dev_q) <= set(scores), "scores không phủ hết dev300"

n_deep = sum(1 for q in dev_q for d in scores[q] if "ce_deep" in scores[q][d])
print(f"{len(dev_q)} câu | doc đã có ce_deep: {n_deep:,} (mong đợi {len(dev_q)*M_OLD:,})")

## Bước 2 — Load reranker

In [ ]:
score_fn = load_reranker(RERANKER_MODEL, device=DEVICE)

## Bước 3 — Chấm hạng 11-20

`skip=M_OLD` bỏ qua đúng phần đã chấm. `ce_deep` cũ được giữ nguyên vì `deepen_one`
copy dict chứ không dựng lại, và thứ tự vẫn sort theo `ce` (không bao giờ bị ghi đè)
nên hạng không xê dịch giữa hai lượt.

In [ ]:
n2 = DC.count_deep_chunks(dev_q, scores, CTX_DIR, m=M_NEW, k=K_CHUNK, skip=M_OLD)
print(f"sẽ chấm {n2:,} đoạn ({n2/len(dev_q):.0f}/câu) — mong đợi ~57.600")
assert n2 < 100_000, "quá nhiều, kiểm lại M_NEW/K_CHUNK"

t0 = time.time()
scores = DC.deepen_all(dev_q, scores, CTX_DIR, score_fn,
                       m=M_NEW, k=K_CHUNK, skip=M_OLD, every=25)
print(f"xong trong {(time.time()-t0)/60:.0f} phút")

n_deep2 = sum(1 for q in dev_q for d in scores[q] if "ce_deep" in scores[q][d])
print(f"doc có ce_deep: {n_deep:,} -> {n_deep2:,} (mong đợi ~{len(dev_q)*M_NEW:,})")

p = f"{OUTPUT_DIR}/scores_dev300_deep_M{M_NEW}_K{K_CHUNK}.json"
json.dump(scores, open(p, "w", encoding="utf-8"), ensure_ascii=False)
print(f"ĐÃ LƯU {p} — TẢI VỀ")

## Bước 4 — Đo

`base` phải vẫn ra **0.8883** ở n=2 — chốt tự kiểm, `ce` không được đụng tới.
So mốc cần vượt là **0.9017** của M=10, không phải 0.8883.

In [ ]:
bm25 = {q: [str(c["doc_id"]) for c in cand[q]] for q in dev_q}

def rec(variant, n):
    pred = {q: blend_bm25_first(DC.rank_by(scores[q], variant), bm25[q], k=TOPK, n_bm25=n)
            for q in dev_q}
    return evaluate(dev_gold, pred, k=TOPK)["recall"]

tab = {v: {n: rec(v, n) for n in (0, 1, 2, 3)} for v in DC.VARIANTS}
print(f"{'biến thể':10s}" + "".join(f"  n={n}    " for n in (0,1,2,3)))
for v, row in tab.items():
    print(f"{v:10s}" + "".join(f"  {row[n]:.4f} " for n in (0,1,2,3)))

chk = tab["base"][2]
print(f"\nchốt tự kiểm: base n=2 = {chk:.4f} (phải {BASE_LINE})"
      + ("  OK" if abs(chk-BASE_LINE) < 0.002 else "  <-- LỆCH, dừng lại"))

bv, bn = max(((v,n) for v in tab for n in tab[v]), key=lambda x: tab[x[0]][x[1]])
best = tab[bv][bn]
print(f"\nM=20 tốt nhất : {bv} n={bn} -> {best:.4f}")
print(f"M=10 mốc cũ   : max n=2 -> {M10_BEST:.4f}")
print(f"chênh lệch    : {best-M10_BEST:+.4f}")
print("\n=> " + ("M=20 THẮNG. Dùng M_DOC=20 trong finalAnswer_run."
                 if best > M10_BEST + 0.005 else
                 "M=20 KHÔNG hơn. Giữ M_DOC=10 — rẻ hơn 2x khi chạy đề thi."))

## Bước 5 — Lưu

In [ ]:
json.dump({"table": tab, "base_line": BASE_LINE, "m10_best": M10_BEST,
           "best": [bv, bn, best], "delta_vs_m10": best - M10_BEST,
           "M_NEW": M_NEW, "K_CHUNK": K_CHUNK, "n_chunk_them": n2},
          open(f"{OUTPUT_DIR}/deepchunk_M20_eval.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}  {os.path.getsize(os.path.join(OUTPUT_DIR,f)):,} bytes")
print("\nTẢI VỀ TOÀN BỘ outputs/")